### Importações

In [2]:
import os
import re
import sys
# Adiciona o diretório pai (raiz do projeto) ao sys.path
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import polars as pl
#from preprocessing import pipeline as pipel

import losses
import joblib

from IPython.terminal import embed

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel

from torch.utils.data import DataLoader
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torch.optim import AdamW

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

model_name = "neuralmind/bert-base-portuguese-cased"

### Preparação dos Dados

In [3]:
dataf = pl.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\dataset_chunks.parquet")
df = dataf.to_pandas()

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 632021 entries, 0 to 632020
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   texto_chunk   632021 non-null  str  
 1   label         632021 non-null  str  
 2   tipo          632021 non-null  str  
 3   chunk_index   632021 non-null  int64
 4   total_chunks  632021 non-null  int64
 5   n_tokens      632021 non-null  int64
dtypes: int64(3), str(3)
memory usage: 1.1 GB


In [5]:
_INTEIRO_TEOR_PATTERNS = [
    (re.compile(r"(Erro Parser)?>>>>>inicio<<<<<\n?", re.MULTILINE | re.IGNORECASE), ""),
    (re.compile(r"fimid:\d+|#####fim#####id:\d+\n?", re.MULTILINE), ""),
]
_MULTI_SPACE = re.compile(r" {2,}")

def limpar_inteiro_teor(text: str) -> str:
    for pattern, replacement in _INTEIRO_TEOR_PATTERNS:
        text = pattern.sub(replacement, text)
    return _MULTI_SPACE.sub(" ", text).strip()


df = df.reset_index(drop=True)
df["id_peticao"] = (df["chunk_index"] == 0).cumsum()

df = df.dropna(subset=["texto_chunk"]).copy()
df["texto_chunk"] = df["texto_chunk"].map(limpar_inteiro_teor)

In [6]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["label"])

num_labels = df["label"].nunique()

print("Número de classes:", num_labels)

Número de classes: 159


### Estratégia de Truncamento

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

ids_peticoes = df["id_peticao"].unique()

train_ids, test_ids = train_test_split(
    ids_peticoes,
    test_size=0.2,
    random_state=42
)

df_train = df[df["id_peticao"].isin(train_ids)].copy()
df_test = df[df["id_peticao"].isin(test_ids)].copy()

dataset_train = Dataset.from_pandas(
    df_train[["texto_chunk", "label"]],
    preserve_index=False
)

dataset_test = Dataset.from_pandas(
    df_test[["texto_chunk", "label"]],
    preserve_index=False
)

dataset_split = DatasetDict({
    "train": dataset_train,
    "test": dataset_test
})



In [ ]:
dataset_split

DatasetDict({
    train: Dataset({
        features: ['texto_chunk', 'label'],
        num_rows: 500415
    })
    test: Dataset({
        features: ['texto_chunk', 'label'],
        num_rows: 131606
    })
})

In [ ]:
def tokenizar(batch):
    return tokenizer(
        batch["texto_chunk"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

dataset_tokenizado = dataset_split.map(
    tokenizar,
    batched=True
)

dataset_tokenizado = dataset_tokenizado.remove_columns(["texto_chunk"])

dataset_tokenizado.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/500415 [00:00<?, ? examples/s]

Map: 100%|██████████| 131606/131606 [01:58<00:00, 1114.22 examples/s]


In [ ]:
batch_size = 512

train_dataloader = DataLoader(
    dataset_tokenizado["train"],
    batch_size=batch_size,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset_tokenizado["test"],
    batch_size=batch_size,
    shuffle=False
)

### Modelo BERT + Projection Head

In [ ]:
class BertSupConModel(nn.Module):
    def __init__(self, model_name, projection_dim=128):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        hidden_size = self.bert.config.hidden_size

        self.projection_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, projection_dim)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        projected_embedding = self.projection_head(cls_embedding)

        projected_embedding = nn.functional.normalize(
            projected_embedding,
            p=2,
            dim=1
        )

        return projected_embedding

### Treinamento : BERT + Projection Head + SupCon Loss

In [ ]:
model_supcon = BertSupConModel(
    model_name=model_name,
    projection_dim=128
)

loss_fn = losses.SupConLoss()

optimizer = AdamW(model_supcon.parameters(), lr=2e-5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_supcon.to(device)

epoch_supcon = 2

model_supcon.train()

for epoch in range(epoch_supcon):
    total_loss = 0

    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        embeddings_1 = model_supcon(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        embeddings_2 = model_supcon(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        features = torch.stack(
            [embeddings_1, embeddings_2],
            dim=1
        )

        loss = loss_fn(features, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_dataloader)

    print(f"Epoch {epoch + 1} | Loss SupCon: {avg_loss:.4f}")

: 

### Remove o Projection Head e Cria o Classificador

In [ ]:
bert_treinado = model_supcon.bert # Remove o projection head com o SupCon Loss e preserva BERT treinado

class BertClassifier(nn.Module):
    def __init__(self, bert, num_labels):
        super().__init__()

        self.bert = bert

        hidden_size = self.bert.config.hidden_size

        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(cls_embedding)

        return logits

### Treinar classificação com o Cross-Entropy

BERT treinado com SupCon + Classifier Head → Cross-Entropy

In [ ]:
model_classifier = BertClassifier(
    bert=bert_treinado,
    num_labels=num_labels
)

model_classifier.to(device)

loss_fn_cls = nn.CrossEntropyLoss()

optimizer = AdamW(model_classifier.parameters(), lr=2e-5)

epoch_classifier = 3 

model_classifier.train()

for epoch in range(epoch_classifier):
    total_loss = 0

    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model_classifier(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn_cls(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)

    print(f"Epoch {epoch + 1} | Loss CE: {avg_loss:.4f}")

### Avaliação

In [ ]:
model_classifier.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model_classifier(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
f1_macro = f1_score(all_labels, all_preds, average="macro")
f1_weighted = f1_score(all_labels, all_preds, average="weighted")

print("Accuracy:", accuracy)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print(classification_report(all_labels, all_preds))

### Salvar modelo

In [ ]:
model_classifier.bert.save_pretrained("bert_supcon_ajustado")
tokenizer.save_pretrained("bert_supcon_ajustado")

torch.save(
    model_classifier.classifier.state_dict(),
    "classifier_head.pt"
)

joblib.dump(
    label_encoder,
    "label_encoder.joblib"
)

### Modelo para inferência

In [ ]:
bert = AutoModel.from_pretrained("bert_supcon_ajustado")
tokenizer = AutoTokenizer.from_pretrained("bert_supcon_ajustado")

label_encoder = joblib.load("label_encoder.joblib")

num_labels = len(label_encoder.classes_)

model_classifier = BertClassifier(
    bert=bert,
    num_labels=num_labels
)

model_classifier.classifier.load_state_dict(
    torch.load("classifier_head.pt", map_location=device)
)

model_classifier.to(device)
model_classifier.eval()

In [ ]:
def prever_tema(texto: str, max_length: int = 512):
    inputs = tokenizer(
        texto,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        logits = model_classifier(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilidades = torch.softmax(logits, dim=1)

        classe_predita_id = torch.argmax(probabilidades, dim=1).item()

        confianca = probabilidades[0, classe_predita_id].item()

    classe_original = label_encoder.inverse_transform(
        [classe_predita_id]
    )[0]

    return {
        "classe_predita": classe_original,
        "classe_id": classe_predita_id,
        "confianca": confianca
    }

In [ ]:
texto = """
Trata-se de ação indenizatória em que a parte autora pleiteia reparação
por danos morais decorrentes de inscrição indevida em cadastro de inadimplentes.
"""

resultado = prever_tema(texto)

print(resultado)

### Função para prever os TOP 3 temas

In [ ]:
def prever_top_k(texto: str, k: int = 3, max_length: int = 512):
    inputs = tokenizer(
        texto,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        logits = model_classifier(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilidades = torch.softmax(logits, dim=1)

        top_probs, top_indices = torch.topk(probabilidades, k=k, dim=1)

    resultados = []

    for prob, idx in zip(top_probs[0], top_indices[0]):
        classe_id = idx.item()
        classe_original = label_encoder.inverse_transform([classe_id])[0]

        resultados.append({
            "classe": classe_original,
            "classe_id": classe_id,
            "confianca": prob.item()
        })

    return resultados

In [ ]:
resultado = prever_top_k(texto, k=3)

for item in resultado:
    print(item)

Accuracy: 0.42301087578706353

F1 Macro: 0.2768311466667517

F1 Weighted: 0.37610847851445556

/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
              precision    recall  f1-score   support

           0       0.25      0.27      0.26        11
           1       0.50      0.11      0.18         9
           2       0.00      0.00      0.00         7
           4       0.36      0.26      0.30        31
           5       0.73      1.00      0.84        16
           6       0.65      1.00      0.79        17
           7       0.91      0.95      0.93        22
           8       0.62      1.00      0.77         5
           9       0.00      0.00      0.00         5
          10       0.48      0.86      0.62        14
          11       0.83      0.83      0.83        24
          12       1.00      0.57      0.73         7
          13       0.38      0.96      0.54        23
          14       0.50      0.50      0.50         2
          15       0.00      0.00      0.00         3
          16       0.00      0.00      0.00         3
          17       0.77      0.94      0.85        18
          18       0.26      0.71      0.38        17
          19       0.18      0.18      0.18        17
          20       0.56      0.79      0.66        29
          21       0.00      0.00      0.00         5
          22       0.21      0.75      0.33        12
          23       0.25      0.81      0.38        21
          24       0.00      0.00      0.00         6
          25       0.67      0.40      0.50         5
          26       0.56      1.00      0.72        14
          27       0.90      0.90      0.90        21
          28       0.23      0.16      0.19        19
          29       0.00      0.00      0.00         2
          30       0.00      0.00      0.00         4
          31       1.00      1.00      1.00         2
          32       0.00      0.00      0.00        17
          33       0.38      0.13      0.19        23
          34       0.21      0.17      0.19        24
          35       0.14      0.09      0.11        23
          36       0.00      0.00      0.00         4
          37       0.00      0.00      0.00         3
          38       0.00      0.00      0.00         4
          39       0.33      0.50      0.40         2
          40       0.00      0.00      0.00         2
          41       0.47      0.56      0.51        16
          42       0.00      0.00      0.00         2
          43       1.00      0.80      0.89         5
          44       0.70      0.83      0.76        23
          45       0.00      0.00      0.00         3
          46       0.24      0.26      0.25        19
          47       0.62      0.88      0.73        17
          48       0.00      0.00      0.00         4
          49       0.50      0.11      0.18         9
          50       1.00      0.31      0.48        16
          51       0.00      0.00      0.00        15
          52       0.00      0.00      0.00         3
          53       0.00      0.00      0.00         3
          54       0.00      0.00      0.00         2
          55       0.00      0.00      0.00         8
          56       0.61      0.65      0.63        17
          57       0.75      0.57      0.65        21
          58       0.86      0.50      0.63        24
          59       0.00      0.00      0.00         5
          60       0.00      0.00      0.00         5
          61       0.56      0.77      0.65        13
          62       0.00      0.00      0.00         2
          63       0.00      0.00      0.00         3
          64       0.00      0.00      0.00        10
          65       0.00      0.00      0.00        10
          66       0.11      0.57      0.19         7
          67       0.00      0.00      0.00         4
          68       0.14      0.29      0.19         7
          69       0.07      0.20      0.10         5
          70       0.50      0.12      0.20         8
          71       0.00      0.00      0.00         8
          72       0.39      0.80      0.52        20
          73       0.50      0.19      0.27        16
          74       0.56      0.75      0.64        20
          75       0.42      0.62      0.50         8
          76       0.58      0.64      0.61        28
          77       0.00      0.00      0.00         3
          78       0.50      0.75      0.60        12
          79       0.25      0.27      0.26        15
          80       0.25      0.11      0.15         9
          81       0.33      0.25      0.29         8
          82       0.00      0.00      0.00         3
          84       0.42      0.56      0.48         9
          85       0.48      0.68      0.57        22
          86       0.41      0.26      0.32        27
          87       0.67      0.62      0.65        16
          88       0.44      0.47      0.46        17
          89       0.42      0.20      0.27        25
          90       0.00      0.00      0.00         6
          91       0.38      0.42      0.40        26
          92       0.00      0.00      0.00         5
          93       0.00      0.00      0.00         2
          94       0.80      0.80      0.80         5
          95       0.00      0.00      0.00         3
          96       0.64      0.78      0.70        18
          97       0.00      0.00      0.00         4
          98       0.00      0.00      0.00         4
          99       0.00      0.00      0.00         3
         100       0.18      0.12      0.15        16
         101       0.00      0.00      0.00         2
         102       0.61      0.73      0.67        15
         103       0.00      0.00      0.00        18
         104       0.85      1.00      0.92        11
         105       0.61      0.64      0.62        22
         106       0.00      0.00      0.00         2
         107       1.00      0.67      0.80         6
         108       0.31      0.56      0.40        16
         109       0.00      0.00      0.00         4
         110       0.89      0.94      0.91        17
         111       0.52      0.80      0.63        15
         112       0.00      0.00      0.00         7
         113       0.00      0.00      0.00         2
         114       0.00      0.00      0.00         5
         115       0.71      0.75      0.73        20
         116       0.00      0.00      0.00         7
         117       0.00      0.00      0.00         3
         118       0.80      1.00      0.89         4
         119       0.62      0.22      0.32        23
         120       0.08      0.67      0.14        12
         121       0.00      0.00      0.00        10
         122       0.00      0.00      0.00        15
         123       0.00      0.00      0.00        18
         124       0.00      0.00      0.00        20
         125       0.00      0.00      0.00         4
         126       0.56      0.62      0.59        16
         127       0.00      0.00      0.00         3
         128       0.00      0.00      0.00         7
         129       0.00      0.00      0.00         6
         130       0.33      0.67      0.44         9
         131       0.00      0.00      0.00         6
         132       0.11      0.27      0.16        11
         133       0.00      0.00      0.00         2
         134       1.00      0.33      0.50         3
         135       0.00      0.00      0.00         4
         136       0.00      0.00      0.00         3
         137       0.41      0.82      0.55        11
         138       0.00      0.00      0.00         4
         139       0.00      0.00      0.00        12
         140       0.14      0.29      0.19        14
         141       0.43      0.20      0.27        15
         142       0.84      1.00      0.91        16
         143       0.00      0.00      0.00         9
         144       0.38      0.15      0.21        20
         145       0.10      0.31      0.15        13
         146       0.00      0.00      0.00        21
         147       0.64      0.88      0.74        16
         148       0.44      0.36      0.40        22
         149       0.57      0.50      0.53         8
         150       0.50      0.10      0.17        10
         151       0.00      0.00      0.00        10
         152       0.41      0.55      0.47        22
         153       0.00      0.00      0.00        11
         154       0.00      0.00      0.00         8
         155       0.15      0.43      0.22        14
         156       0.00      0.00      0.00         2
         157       0.00      0.00      0.00         1
         158       0.00      0.00      0.00         3

accuracy                           0.42      1747\
macro avg       0.28      0.31      0.28      1747\
weighted avg       0.38      0.42      0.38      1747

{'classe_predita': 'STJ_1156', 'classe_id': 87, 'confianca': 0.18051281571388245}\
{'classe': 'STJ_1156', 'classe_id': 87, 'confianca': 0.18051281571388245}\
{'classe': 'STF_1417', 'classe_id': 26, 'confianca': 0.0444086529314518}\
{'classe': 'STJ_414', 'classe_id': 126, 'confianca': 0.039152733981609344}